# 05 · Spark ALS, temporal evaluation and cold start

BDA Units 2–4. Train distributed collaborative filtering on explicit MovieLens ratings, export item factors, and combine them with genre/theme content. Use Java 17 and `python -m pip install -e ".[spark,notebooks]"`.

In [1]:
from pathlib import Path
import sys, json, os
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists(): ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), 'Open this notebook from CineScope/notebooks or CineScope.'
sys.path.insert(0, str(ROOT / 'backend'))
sys.path.insert(0, str(ROOT / 'scripts'))
os.chdir(ROOT)
(ROOT / 'artifacts').mkdir(exist_ok=True)
SEED = 42

In [2]:
RUN_SPARK_TRAINING = False
ML_DIR = ROOT/'data/raw/ml-latest-small'
if RUN_SPARK_TRAINING:
    from pyspark.sql import SparkSession,Window,functions as F,types as T
    from pyspark.ml.recommendation import ALS
    from pyspark.ml.evaluation import RegressionEvaluator
    spark = SparkSession.builder.master('local[2]').appName('CineScope Notebook').config('spark.sql.shuffle.partitions','8').getOrCreate()
    schema=T.StructType([T.StructField('userId',T.IntegerType()),T.StructField('movieId',T.IntegerType()),T.StructField('rating',T.DoubleType()),T.StructField('timestamp',T.LongType())])
    ratings=spark.read.option('header',True).schema(schema).csv(str(ML_DIR/'ratings.csv')).dropna()
    print('Partitions:',ratings.rdd.getNumPartitions())
    ratings.groupBy('movieId').count().explain(True)
    ratings.show(5)
else:
    print('Spark training disabled. This notebook still demonstrates the hybrid serving path below.')

Spark training disabled. This notebook still demonstrates the hybrid serving path below.


In [3]:
if RUN_SPARK_TRAINING:
    by_user=Window.partitionBy('userId')
    chronology=by_user.orderBy('timestamp','movieId')
    ordered=ratings.withColumn('rn',F.row_number().over(chronology)).withColumn('n',F.count('*').over(by_user)).filter('n >= 5')
    train=ordered.filter(F.col('rn')<=F.floor(F.col('n')*.8)).cache()
    valid=ordered.filter((F.col('rn')>F.floor(F.col('n')*.8))&(F.col('rn')<=F.floor(F.col('n')*.9))).cache()
    test=ordered.filter(F.col('rn')>F.floor(F.col('n')*.9)).cache()
    evaluator=RegressionEvaluator(labelCol='rating',predictionCol='prediction',metricName='rmse')
    trials=[]
    for rank in (16,32):
        for reg in (.05,.1):
            model=ALS(userCol='userId',itemCol='movieId',ratingCol='rating',rank=rank,regParam=reg,maxIter=12,coldStartStrategy='drop',seed=SEED).fit(train)
            predictions=model.transform(valid)
            trials.append((evaluator.evaluate(predictions),rank,reg,model))
    validation_rmse,rank,reg,model=min(trials,key=lambda x:x[0])
    test_predictions=model.transform(test).cache()
    report={'validation_rmse':validation_rmse,'test_rmse':evaluator.evaluate(test_predictions),
            'test_coverage':test_predictions.count()/test.count(),'rank':rank,'reg_param':reg}
    print(report)
    (ROOT/'artifacts/als_metrics.json').write_text(json.dumps(report,indent=2))

In [4]:
if RUN_SPARK_TRAINING:
    import numpy as np
    model.write().overwrite().save(str(ROOT/'models/spark_als'))
    model.itemFactors.write.mode('overwrite').parquet(str(ROOT/'models/item_factors'))
    # Only the compact factor table is collected for single-process serving.
    factors=model.itemFactors.orderBy('id').toPandas()
    np.savez_compressed(ROOT/'models/als_factors.npz',movie_ids=factors.id.to_numpy(dtype=np.int64),
                        factors=np.stack(factors.features.to_numpy()).astype(np.float32),reg_param=np.array(reg))
    candidates=model.recommendForAllUsers(100).select('userId',F.explode('recommendations').alias('rec')).select('userId',F.col('rec.movieId').alias('movieId'),F.col('rec.rating').alias('score'))
    unseen=candidates.join(train.select('userId','movieId'),['userId','movieId'],'left_anti')
    ranked=unseen.withColumn('rank',F.row_number().over(Window.partitionBy('userId').orderBy(F.desc('score'),F.asc('movieId')))).filter('rank <= 10')
    ranked.write.mode('overwrite').parquet(str(ROOT/'artifacts/als_ranked'))
    test.select('userId','movieId','rating').write.mode('overwrite').parquet(str(ROOT/'artifacts/als_test'))
    train.select('userId','movieId','rating').write.mode('overwrite').parquet(str(ROOT/'artifacts/als_train'))
    spark.stop()

## Hybrid ranking
The runtime uses a signed, recency-weighted feature profile: high ratings add affinity and low ratings subtract it. For known ALS items it solves a regularized least-squares fold-in vector from the live user's ratings. It then blends collaborative scores with content/query similarity and a weak prior when available, excludes watched films, and applies a diversity penalty.

The demo catalog IDs are isolated from MovieLens IDs. ALS factors are deliberately ignored for demo data. New movies use content features; new users receive exploration suggestions until they rate films. A Sci-Fi+Romance request uses an intersection of genres, not an OR.

In [5]:
from cinescope.recommender import Recommender
movies=json.loads((ROOT/'data/demo/catalog.json').read_text())
recommender=Recommender(movies)
history=[{'movie_id':3,'rating':5},{'movie_id':9,'rating':4.5},{'movie_id':10,'rating':1}]
picks=recommender.recommend(history,genres=['Sci-Fi','Romance'])
[(m['title'],m['rank_score'],m['reasons'],m['cold_start']) for m in picks]

[('Eternal Sunshine of the Spotless Mind', 0.1198, ['Matches your highly rated Sci-Fi + Romance + Drama films'], True), ('The Last Signal', 0.0725, ['Matches your highly rated Sci-Fi + Romance + Love films'], True)]


## Evaluation caveats
Report RMSE **and** cold-start coverage. Dropping NaN ALS predictions without reporting the dropped fraction hides cold-start failures. For ranking use recall@K, NDCG@K, precision@K, item coverage and diversity. The Spark exports overfetch 100 candidates before seen-item removal; this can produce fewer than K results for prolific users. Record that candidate cap and increase it or score candidates in batches for a stricter ranking benchmark.

Notebook 07 compares ALS to a popularity baseline fitted on train interactions only. Tune hybrid weights on validation interactions, never on test. Compare performance separately for sparse users and new items.